# 01 — EDA on the CFPB Consumer Complaint Database

Quick exploration before building the Tableau dashboard. Goal: understand the shape, the volume by year, the heavy-hitter products and companies, and the resolution-outcome distribution.

Fill in the TODOs, run the cells, write your observations in the markdown blocks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

df = pd.read_parquet('../prepared/complaints.parquet')
print(f'{len(df):,} rows, {len(df.columns)} columns')
print(f'Date range: {df["date_received"].min().date()} -> {df["date_received"].max().date()}')

## Schema and missing data

In [ ]:
df.info()
print('\nMissing per column:')
print(df.isna().sum().sort_values(ascending=False))

**Observations**

_(What's heavily missing? Sub-product and sub-issue are often null — expected, those are drill-downs that don't always apply. consumer_disputed should be largely null post-2017 because CFPB stopped collecting it.)_

## Volume over time

In [ ]:
monthly = df.set_index('date_received').resample('MS').size()
monthly.plot(title='Complaints per month', ylabel='count')
plt.show()
print(f'Peak month: {monthly.idxmax().date()} with {monthly.max():,} complaints')

**Observations**

_(Is there a clear trend? Any seasonality? Big spikes around specific events e.g. COVID, regulatory enforcement actions?)_

## Top products

In [ ]:
products = df['product'].value_counts()
print(products.head(15).to_string())
products.head(10).plot(kind='barh', title='Top 10 products by complaint volume', figsize=(10, 5))
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Observations**

_(Credit reporting will likely dominate. EFT / debit-card / prepaid-card complaints are the Reg-E story — worth tracking specifically.)_

## Top companies

In [ ]:
companies = df['company'].value_counts()
print(companies.head(20).to_string())

**Observations**

_(Equifax / Experian / TransUnion will dominate because credit-reporting complaints route to them. Big banks — BofA, Wells, Chase, Capital One — should also be near the top. PNC's relative position vs peers is worth knowing for the dashboard.)_

## Resolution outcomes

In [ ]:
responses = df['company_response'].value_counts()
print(responses.to_string())
# Fraction with monetary or non-monetary relief
relief = df['company_response'].isin(['Closed with monetary relief', 'Closed with non-monetary relief']).mean()
print(f'\n{relief:.1%} of complaints close with some form of relief')

**Observations**

_(Most complaints close with explanation only. Relief rate varies meaningfully by product and company — that's the story for the dashboard's resolution panel.)_

## Geographic distribution

In [ ]:
states = df['state'].value_counts()
print(states.head(15).to_string())
# Per-capita normalization will need state population data — for now just raw counts

**Observations**

_(CA/TX/FL/NY are biggest by raw count — mostly population effect. The dashboard should normalize per capita to surface real geographic patterns.)_

## Reg-E specific slice

The complaints that map to your day job at PNC: prepaid card, money transfers, debit card EFT issues.

In [ ]:
rege_products = ['Money transfer, virtual currency, or money service',
                 'Prepaid card',
                 'Checking or savings account',
                 'Bank account or service']
rege_df = df[df['product'].isin(rege_products)]
print(f'{len(rege_df):,} Reg-E-adjacent complaints ({len(rege_df) / len(df):.1%} of all)')
rege_monthly = rege_df.set_index('date_received').resample('MS').size()
rege_monthly.plot(title='Reg-E-adjacent complaints over time', ylabel='count')
plt.show()

**Observations**

_(Is this category growing? Stable? That's the headline story for the portfolio writeup.)_

## Bottom line

_(What three insights would you put on the Tableau dashboard headline panel? What's the most surprising thing you found?)_